# CSA0812 – Assignment 5
## A1 – Design the Data Acquisition

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Outcome:** CO5  
**Marks:** 7 | **K3 – Apply**

**Aim:** Understand the daily air-quality CSV files, decide how to store the readings, and compare CSV files with a simple SQLite database.

**Beginner note:** This is a design task. Most of the work is explaining the plan. The small SQL examples show what the database would do; you do not need to build a full database program for A1.

## A1.1 – Inspect the Source (2 Marks)

Each station sends a CSV file every day. A CSV file is a text file arranged in rows and columns. Each row stores one reading from one station at one date and hour.

| Column | Simple meaning | Example |
|---|---|---|
| station | Station name | Adyar |
| date | Reading date | 2026-10-01 |
| hour | Hour of the day (0–23) | 6 |
| pm25 | PM2.5 pollution reading | 48.5 |
| pm10 | PM10 pollution reading | 92.0 |
| no2 | Nitrogen dioxide reading | 18.2 |
| temperature | Temperature value | 77 |
| temperature_unit | Celsius or Fahrenheit | F or C |

### Problems that can occur

| Problem | Example | Why it is a problem |
|---|---|---|
| Missing reading | blank or `NA` | Cannot use it directly in an average |
| Negative reading | PM10 = -5 | A negative pollution reading is not valid |
| Very large reading | PM2.5 = 1500 | Could be a sensor fault |
| Duplicate row | Same station/date/hour appears twice | The reading may be counted twice |
| Wrong date | 31/19/2026 | Cannot be sorted correctly |
| Wrong hour | 25 | Hours must be from 0 to 23 |
| Different station spelling | adyar and Adyar | One station may be treated as two stations |
| Different temperature units | 77°F and 25°C | Values cannot be compared directly |
| Wrong file format | Semicolon instead of comma | Columns may not be separated correctly |

**Answer:** The files may contain missing values, invalid or unusually large readings, duplicate rows, wrong dates or hours, inconsistent station names, different temperature units and damaged file formats. The program must check these problems before analysis.

## A1.2 – Design the Storage (3 Marks)

The board can store the cleaned data in a small SQLite database. A database is like a collection of organised tables. A table looks similar to a spreadsheet: it has columns and rows.

We can use **two tables**.

### Table 1: `stations`

| Column | Type | Meaning |
|---|---|---|
| station_id | INTEGER | Unique number for each station |
| station_name | TEXT | Name such as Adyar |
| temperature_unit | TEXT | Original unit sent by the station |

### Table 2: `readings`

| Column | Type | Meaning |
|---|---|---|
| reading_id | INTEGER | Unique number for each reading |
| station_id | INTEGER | Connects the reading to its station |
| date | TEXT | Date in YYYY-MM-DD format |
| hour | INTEGER | Hour from 0 to 23 |
| pm25 | REAL | PM2.5 decimal value |
| pm10 | REAL | PM10 decimal value |
| no2 | REAL | NO₂ decimal value |
| temperature_c | REAL | Temperature converted to Celsius |
| source_file | TEXT | Name of the original CSV file |

### Simple database words

- **Table:** A place to store related information, like a spreadsheet.
- **Column:** One type of information, such as station name.
- **Row:** One saved record.
- **Primary key:** A unique ID for each row.
- **Foreign key:** A value that connects one table to another.
- **TEXT:** Used for words or dates.
- **INTEGER:** Used for whole numbers.
- **REAL:** Used for decimal numbers.

### Simple table design example

```sql
CREATE TABLE stations (
    station_id INTEGER PRIMARY KEY,
    station_name TEXT,
    temperature_unit TEXT
);

CREATE TABLE readings (
    reading_id INTEGER PRIMARY KEY,
    station_id INTEGER,
    date TEXT,
    hour INTEGER,
    pm25 REAL,
    pm10 REAL,
    no2 REAL,
    temperature_c REAL,
    source_file TEXT
);
```

You do not need to memorise this. It shows the two tables and their columns.

### How the CSV columns are stored

| CSV column | Database column |
|---|---|
| station | Find the station in `stations`; use its station_id in `readings` |
| date | readings.date |
| hour | readings.hour |
| pm25 | readings.pm25 |
| pm10 | readings.pm10 |
| no2 | readings.no2 |
| temperature | Convert to Celsius and save as readings.temperature_c |
| temperature_unit | stations.temperature_unit |
| CSV filename | readings.source_file |

**Why two tables?** The station name and its original temperature unit do not need to be repeated for every reading. The station ID connects each reading to the correct station.

## A1.3 – CSV or SQL? (2 Marks)

First, remember:

- **CSV:** A file that looks like a simple spreadsheet.
- **SQL:** A language used to ask questions about information in a database.
- **SQLite:** A small database system that can be used from Python.

### Question 1: What is the average PM2.5 at Adyar during October?

**Using CSV:** Open the CSV files, find the Adyar rows from October, and calculate the average PM2.5.

**Using SQL:**

```sql
SELECT AVG(pm25)
FROM readings
WHERE station_id = 1
  AND date >= '2026-10-01'
  AND date < '2026-11-01';
```

- `SELECT` means choose the result.
- `AVG(pm25)` means calculate the average of the PM2.5 column.
- `FROM readings` means use the readings table.
- `WHERE` means only use rows that match a condition.

The example uses station ID 1 for Adyar. The actual program would use the correct station ID and October date range.

### Question 2: What is the highest PM10 reading?

**Using CSV:** Read each row and keep the largest PM10 number.

**Using SQL:**

```sql
SELECT station_id, pm10
FROM readings
ORDER BY pm10 DESC
LIMIT 1;
```

`ORDER BY pm10 DESC` puts the highest PM10 value first, and `LIMIT 1` shows only that top row.

### Question 3: How many PM2.5 readings above 100 did each station have?

**Using CSV:** Read the rows and count values above 100 for each station.

**Using SQL:**

```sql
SELECT station_id, COUNT(*)
FROM readings
WHERE pm25 > 100
GROUP BY station_id;
```

- `COUNT(*)` counts rows.
- `pm25 > 100` keeps readings above 100.
- `GROUP BY station_id` makes a separate count for each station.

### Simple comparison

| CSV files | SQLite database |
|---|---|
| Easy to open and view | Data is saved in organised tables |
| Good for receiving daily files | Good for asking repeated questions |
| May need to read all files again for each question | Can search using short SQL commands |
| Simple, but repeated searches take longer as files grow | Can search faster when an index is added |

**Conclusion:** CSV is useful for receiving the original data. SQLite is useful when the board needs to ask many questions about the same data.

## Alternative Concept and Complexity Analysis

Another option is to keep the data only in CSV files and use pandas to open the files every time the board asks a question. The program must read the file, find the required rows and calculate the answer again for each question.

Let `n` mean the total number of readings.

| Point | CSV + pandas | SQLite with an index |
|---|---|---|
| Time for one question | May need to check every row: O(n) | An indexed search is about O(log n), plus the matching rows |
| Repeated questions | May read the same files again | Data is already stored in the database |
| Memory | Loads data into the computer's RAM | Main data stays in the database file; only needed results may be returned |
| Best use | Small tasks or checking a file once | Repeated questions and larger data |

### What does Big-O mean?

- **O(n):** If the number of readings doubles, the program may need to check about twice as many rows.
- **O(log n):** An indexed search can find a location without checking every row one by one. It usually grows more slowly.
- **RAM:** The computer's working memory. A large DataFrame uses more RAM because it loads data into memory.

The assignment's basic comparison is a full CSV scan O(n) versus an indexed search about O(log n). In practice, SQL also needs to return and process the rows that match the search.

### A1 – Final conclusion

Daily CSV files are a simple way for stations to send readings. Before using the data, the program must check for missing values, invalid readings, duplicates and different temperature units. SQLite can store the cleaned information in two connected tables and make repeated searches easier. This design is suitable for the air-quality monitoring network.